In [1]:
# from google.colab import drive
# drive.mount("/content/drive")

In [2]:
import os
import sys
import random
import warnings
import math

import numpy as np
import pandas as pd
import h5py
import cv2
from PIL import Image

import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.metrics import (
    average_precision_score,
    label_ranking_average_precision_score,
    roc_auc_score
)

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler

from torchvision import models

import albumentations as A
from albumentations.core.transforms_interface import ImageOnlyTransform
from albumentations.pytorch import ToTensorV2

# Custom modules (Kaggle inputs)
#sys.path.append("/content/drive/MyDrive/BHF_Data")
from preprocess import (
    ecg_processing_pipeline,
    smart_pad_and_resize_ecg,
    ecg_processing_pipeline_no_perspective_distortion
)
from metrics import (
    batch_training_metrics,
    aggregate_training_epoch,
    compute_ranking_metrics
)


In [3]:
def check_device():
    """
    Check available compute devices and return the best one.
    Priority: CUDA > MPS > CPU
    """
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("✓ CUDA available")
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("✓ MPS (Apple Silicon GPU) available")
    else:
        device = torch.device("cpu")
        print("✗ Using CPU (no GPU acceleration available)")

    print(f"\nSelected device: {device}")
    return device

# Check and get device
device = check_device()

✓ MPS (Apple Silicon GPU) available

Selected device: mps


In [4]:
label_df = pd.read_csv("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/train_final.csv", index_col=0)

In [5]:
class Head(nn.Module):
    def __init__(self, in_features, hidden_layer, dropout_rate=0.3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features, hidden_layer),
            nn.BatchNorm1d(hidden_layer),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_layer, hidden_layer // 2),
            nn.BatchNorm1d(hidden_layer // 2),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_layer // 2, 1)
        )

    def forward(self, x):
        return self.layers(x)

class MultiHeadEfficientNet(nn.Module):
    def __init__(self, num_conditions=5, hidden_dim=512, dropout_rate=0.3):
        super().__init__()

        backbone = models.convnext_base(weights="IMAGENET1K_V1", progress=True)
        in_features = backbone.classifier[2].in_features
        assert isinstance(in_features, int), f"in_features should be int, got {type(in_features)}"

        backbone.classifier = nn.Identity()
        self.backbone = backbone

        self.shared_feature_processor = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.BatchNorm1d(in_features),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_rate)
        )

        self.heads = nn.ModuleList([
            Head(hidden_dim, hidden_dim // 2, dropout_rate)
            for _ in range(num_conditions)
        ])

    def forward(self, x):
        backbone_feats = self.backbone(x).flatten(1) # squeeze non-batch dimension
        processed_feats = self.shared_feature_processor(backbone_feats)
        outputs = [head(processed_feats) for head in self.heads]
        return torch.cat(outputs, dim=1)  # [batch, num_conditions]

In [6]:
# load images in from hdf5 file
def read_preprocessed_images(filepath):

    with h5py.File(filepath, 'r') as f:
        raw_names = f["img_names"][:]
        images = f["images"][:]
    image_names = [n.decode("utf-8") for n in raw_names]
    image_set = dict(zip(image_names, images))

    return image_set

image_set = read_preprocessed_images("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/proc_images.h5")

In [7]:
from sklearn.model_selection import train_test_split

image_names = list(image_set.keys())
image_ids = list([int(image_name.split(".")[0][-6:]) for image_name in image_names])
label_df = label_df.loc[label_df.index.isin(set(image_ids))]

# need to order label_df so that it has the same ordering as image_names
image_df = pd.DataFrame({
    "image_name": image_set.keys(),
})
image_df["image_id"] = image_df["image_name"].str.split(".", expand=True)[0].str[-6:].astype(int)
image_df = image_df.sort_values(by="image_id")
image_df = image_df.set_index("image_id")
# ensure ordering of label_df and image_df
label_df = label_df.loc[image_df.index]
X_train, X_test, y_train, y_test = train_test_split(image_df,
                                                    label_df,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    shuffle=True,
                                                    stratify=label_df[["CD", "MI", "AF", "STTC", "HYP"]]
                                                    )
# Print prevalance to check stratification
print("Raw Data")
print(label_df.mean(axis=0))
print("Train")
print(y_train.mean(axis=0))
print("Test")
print(y_test.mean(axis=0))

train_image_names = X_train["image_name"].tolist()
test_image_names = X_test["image_name"].tolist()

Raw Data
STTC    0.242471
HYP     0.123268
MI      0.254331
CD      0.208955
AF      0.068830
dtype: float64
Train
STTC    0.242545
HYP     0.123272
MI      0.254456
CD      0.209145
AF      0.068882
dtype: float64
Test
STTC    0.242172
HYP     0.123251
MI      0.253831
CD      0.208195
AF      0.068621
dtype: float64


In [8]:
class ECGDataset(Dataset):
    def __init__(
        self,
        image_names,  # list of filenames
        image_set,    # dict: {filename: np.array(H,W)}
        labels_df,    # dataframe indexed by image_id or image_name
        transforms=None
    ):
        assert len(image_names) == len(labels_df), \
        "Mismatch between number of labels and number of images"
        self.image_names = list(image_names)
        self.images = image_set
        self.labels = torch.tensor(labels_df.values, dtype=torch.float32)
        self.transforms = transforms

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]
        image = self.images[image_name]
        # Duplicate grayscale to 3 channels: (H, W, 3)
        if image.ndim == 2:
            image = np.stack([image, image, image], axis=-1)

        if self.transforms is not None:
            image_tens = self.transforms(image=image)["image"]
        else:
            image_tens = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        label = self.labels[idx]
        return image_tens, label


In [9]:
train_transforms = A.Compose([
    A.CLAHE(
        clip_limit=4,
        tile_grid_size=(8, 8),
        p=1.0
    ),
    #Data Augmentations
    A.Rotate(limit=2, p=0.3),  # small rotations, limit is +/- degrees
    A.Affine(translate_percent={'x': (-0.1, 0.1), 'y': (-0.05, 0.05)},
            rotate=0, scale=1.0, shear=0, p=0.3),  # small translations
    A.RandomShadow( # shadows
        shadow_roi=(0, 0, 1, 1),  # Can appear anywhere in image
        num_shadows_limit=(1,2),  # 1-2 shadow regions
        shadow_dimension=4,         # Controls shadow size/complexity
        shadow_intensity_range=(0.2, 0.4),
        p=0.3
    ),
    A.ElasticTransform(
        alpha=30,
        sigma=15,
        interpolation=cv2.INTER_AREA,
        p=0.3
    ),
    A.GaussianBlur(
        blur_limit=0,
        sigma_limit=(0.1, 1.0),
        p=0.3
    ),
    A.Perspective(
        scale=[0.01, 0.03],
        keep_size=True,
        fit_output=True,
        interpolation=cv2.INTER_AREA,
        mask_interpolation=cv2.INTER_AREA,
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        fill_mask=0,
        p=0.3
    ),
    # Normalization (ImageNet)
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    # Convert to tensor
    ToTensorV2()
])

val_transforms = A.Compose([
    A.CLAHE(
        clip_limit=4,
        tile_grid_size=(8, 8),
        p=1.0
    ),
    # Normalization (ImageNet)
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    # Convert to tensor
    ToTensorV2()
])

In [11]:
num_workers = 0 if sys.platform == "darwin" else 2
print(f"Using num_workers = {num_workers}")

train_dataset = ECGDataset(
    image_names = train_image_names,
    image_set = image_set,
    labels_df = y_train,
    transforms = train_transforms,

)
val_dataset = ECGDataset(
    image_names = test_image_names,
    image_set = image_set,
    labels_df = y_test,
    transforms = val_transforms
)

pin_memory = (device.type == "cuda")
prefetch_factor = 2 if num_workers > 1 else 1
persistent_workers = (num_workers > 1)
print(f"Using pin_memory = {pin_memory}")
print(f"Using prefetch_factor = {prefetch_factor}")
print(f"Using persistent_workers = {persistent_workers}")

train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=num_workers,
    #prefetch_factor = prefetch_factor, # has to be commented out if num_workers = 0
    persistent_workers=persistent_workers,
    pin_memory=pin_memory
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=num_workers,
    #prefetch_factor=prefetch_factor,
    persistent_workers=persistent_workers,
    pin_memory=pin_memory
)

Using num_workers = 0
Using pin_memory = False
Using prefetch_factor = 1
Using persistent_workers = False


In [12]:
# load checkpoint
checkpoint_path = "convnext_checkpoint.pth"
has_checkpoint = False
try:
    checkpoint = torch.load(checkpoint_path, map_location=torch.device("cpu"), weights_only=False)
    has_checkpoint = True
    current_epoch = checkpoint["epoch"]
    print(f"Loading from checkpoint, last run epoch was {current_epoch}")
except Exception as e:
    print("No checkpoint found, continuing as default")
    current_epoch = 0

num_epochs_decay = 20
num_epochs_frozen = 3
num_epochs_const = 10
num_warmup_epochs = 3
num_epochs_total = num_epochs_decay + num_epochs_frozen + num_epochs_const + num_warmup_epochs

No checkpoint found, continuing as default


In [13]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Multi-label focal loss with optional per-class alpha and gamma.
    Handles device placement automatically and ensures numerical stability.
    """

    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        """
        Args:
            gamma (float or tensor): focusing parameter; scalar or per-class vector.
            alpha (float or tensor): class balance weights; scalar or per-class vector.
            reduction (str): "mean", "sum", or "none".
        """
        super().__init__()
        self.reduction = reduction

        # ---- Store gamma (scalar or vector) ----
        if torch.is_tensor(gamma):
            self.register_buffer("gamma", gamma.float())
            self.gamma_is_scalar = False
        else:
            self.gamma = float(gamma)
            self.gamma_is_scalar = True

        # ---- Store alpha (scalar or vector) ----
        if alpha is not None:
            if not torch.is_tensor(alpha):
                alpha = torch.tensor(alpha, dtype=torch.float32)
            self.register_buffer("alpha", alpha.float())
            self.alpha_is_set = True
        else:
            self.alpha_is_set = False

    def forward(self, logits, targets):
        """
        Args:
            logits: raw model outputs (batch, num_classes)
            targets: binary labels (batch, num_classes)
        """

        # ---- Numerically stable sigmoid + BCE ----
        # Instead of sigmoid(logits) then BCE, we use the built-in stable function.
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, reduction="none"
        )

        # Stable sigmoid
        p = torch.sigmoid(logits)

        # p_t = p for y=1, else 1-p
        pt = p * targets + (1 - p) * (1 - targets)

        # ---- Make sure gamma and alpha match device and shape ----
        if self.gamma_is_scalar:
            gamma = self.gamma
        else:
            gamma = self.gamma.to(logits.device)  # (num_classes,)

        if self.alpha_is_set:
            alpha = self.alpha.to(logits.device)  # (num_classes,)
            alpha_t = alpha * targets + 1.0 * (1 - targets)
        else:
            alpha_t = 1.0

        # ---- Focal modulation ----
        # Add eps for numerical stability: (1 − pt) never becomes exactly 0
        eps = 1e-8
        focal_weight = (1 - pt + eps) ** gamma

        # ---- Apply alpha weighting ----
        focal_weight = focal_weight * alpha_t

        # ---- Combine focal term with BCE ----
        loss = focal_weight * bce

        # ---- Reduction ----
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss

freq = label_df.mean()
inv_freq = 1.0/freq
alpha = (inv_freq/inv_freq.max()).to_numpy()

criterion = FocalLoss(
    gamma = 2.0,
    alpha = alpha,
    reduction="mean"
)

In [14]:
model = MultiHeadEfficientNet(
    num_conditions=5,
    hidden_dim=512,
    dropout_rate=0.3
).to(device)

if has_checkpoint:
    print("Loading model state dict from checkpoint")
    model.load_state_dict(checkpoint["model_state_dict"])

In [15]:
# initially, we are going to freeze the weights of the backbone and just train new features
if current_epoch < num_epochs_frozen:
    for param in model.backbone.parameters():
        param.requires_grad = False

opt = torch.optim.AdamW(
    [
        {"params": model.backbone.parameters(), "lr": 0.0},
        {"params": model.shared_feature_processor.parameters(), "lr": 1.0e-3},
        {"params": model.heads.parameters(), "lr": 1.0e-3}
    ],
    weight_decay = 1.0e-3
)

if has_checkpoint:
    print("Loading up optimizer state dict")
    opt.load_state_dict(checkpoint["opt_state_dict"])

In [16]:
def get_lr(
        current_epoch,
        frozen_epochs=num_epochs_frozen,
        warmup_epochs=3,
        decay_epochs=80,
        total_epochs=num_epochs_total,
        min_lrs=[1.0e-6, 5.0e-6, 1.0e-5],
        max_lrs=[1.0e-4, 1.0e-3, 1.0e-3],
        ):

    out_lrs = {"backbone": None, "shared": None, "head": None}

    if current_epoch < frozen_epochs:
        out_lrs["backbone"] = 0.0
        out_lrs["shared"] = max_lrs[1]
        out_lrs["head"] = max_lrs[2]
        return out_lrs

    if current_epoch < frozen_epochs + warmup_epochs:
        inv_warmup_epochs = current_epoch - frozen_epochs
        out_lrs["backbone"] = max_lrs[0] * (inv_warmup_epochs / warmup_epochs)
        out_lrs["shared"] = max_lrs[1]
        out_lrs["head"] = max_lrs[2]
        return out_lrs

    if current_epoch < frozen_epochs + warmup_epochs + decay_epochs:
        decay_epochs = frozen_epochs + warmup_epochs + decay_epochs - frozen_epochs - warmup_epochs
        decay_progress = (current_epoch - frozen_epochs - warmup_epochs) / decay_epochs
        decay_progress = min(max(decay_progress, 0), 1)  # clamp
        out_lrs["backbone"] = min_lrs[0] + (max_lrs[0] - min_lrs[0]) * 0.5 * (1 + math.cos(math.pi * decay_progress))
        out_lrs["shared"] = min_lrs[1] + (max_lrs[1] - min_lrs[1]) * 0.5 * (1 + math.cos(math.pi * decay_progress))
        out_lrs["head"] = min_lrs[2] + (max_lrs[2] - min_lrs[2]) * 0.5 * (1 + math.cos(math.pi * decay_progress))
        return out_lrs

    out_lrs["backbone"] = min_lrs[0]
    out_lrs["shared"] = min_lrs[1]
    out_lrs["head"] = min_lrs[2]

    return out_lrs

In [17]:
class EarlyStopping:
    """
    Early stops the training if validation loss doesn't improve after 'patience' epochs.
    Saves the best model automatically.
    """
    def __init__(self, patience=3, verbose=True, delta=0.0, best_loss=None, save_path="best_checkpoint.pth"):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.save_path = save_path

        self.best_loss = best_loss if best_loss is not None else float("inf")
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss, epoch, model, opt, extra_state=None):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0

            # save best model
            checkpoint = {
                "model_state_dict": model.state_dict(),
                "opt_state_dict": opt.state_dict(),
                "epoch": epoch,
            }
            if extra_state:
                checkpoint.update(extra_state)
            torch.save(checkpoint, self.save_path)

            if self.verbose:
                print(f"  ✓ Validation improved → saving new best model (loss={val_loss:.5f})")
        else:
            self.counter += 1
            if self.verbose:
                print(f"  ✗ No improvement ({self.counter}/{self.patience})")
            if self.counter >= self.patience:
                self.early_stop = True


In [18]:
scaler = GradScaler('cuda') if device.type == "cuda" else None
train_losses = []
train_epoch_stats = []
val_losses = []
val_ranking_stats = []

early_stopper = EarlyStopping(
    patience=3,
    verbose=True,
    save_path="best_model.pth",
    best_loss=None
)
label_smoothing_eta = 0.03

def label_smoothing(labels, eta=label_smoothing_eta):
    return labels * (1-eta) + 0.5 * eta

for epoch in range(current_epoch, num_epochs_total):

    if epoch == num_epochs_frozen:
        # unfreeze parameters
        for param in model.backbone.parameters():
            param.requires_grad = True

    # get learning rates for current epoch
    lrs = get_lr(epoch)
    opt.param_groups[0]["lr"] = lrs["backbone"]
    opt.param_groups[1]["lr"] = lrs["shared"]
    opt.param_groups[2]["lr"] = lrs["head"]
    print("="*100)
    print(f"For epoch {epoch}, using learning rates {lrs}")

    model.train()
    opt.zero_grad()
    pbar = tqdm(total=len(train_dataloader),
                desc=f"Epoch {epoch} - Training",
                unit="batch")
    running_train_loss = torch.tensor(0.0, device=device)
    total_samples = 0
    train_batch_stats = []
    for i, (inputs, labels) in enumerate(train_dataloader):
        inputs = inputs.to(device)
        labels = labels.to(device)
        labels_smooth = label_smoothing(labels)

        if device.type == 'cuda':
            with autocast('cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, labels_smooth)
            scaler.scale(loss).backward()
        else:
            # doesn't support mixed precision training
            outputs = model(inputs)
            loss = criterion(outputs, labels_smooth)
            loss.backward()

        with torch.no_grad():
            y_pred = torch.sigmoid(outputs.float())     # convert logits → probabilities
            stats = batch_training_metrics(labels, y_pred)
            train_batch_stats.append(stats)

        if device.type == "cuda":
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            opt.zero_grad()

        pbar.update(1)

        # accumulate metrics
        running_train_loss = running_train_loss + (loss.detach()  * inputs.shape[0])
        total_samples += inputs.shape[0]

    pbar.close()

    avg_train_loss = (running_train_loss / total_samples).item()
    train_losses.append({"epoch": epoch, "avg_train_loss": avg_train_loss})
    print(f"Epoch {epoch} Train Loss: ", avg_train_loss)
    train_stats_epoch = aggregate_training_epoch(train_batch_stats, total_samples)
    print(f"Epoch {epoch} TRAIN MONITOR:", train_stats_epoch)
    train_epoch_stats.append({"epoch": epoch, **train_stats_epoch})

    # VALIDATION LOOP
    model.eval()
    pbar = tqdm(total=len(val_dataloader),
                desc=f"Epoch {epoch} - Validation",
                unit="batch")
    running_val_loss = torch.tensor(0.0, device=device)
    total_samples = 0
    val_y_true_list = []
    val_y_pred_list = []
    with torch.no_grad():
        for i, (inputs, labels) in enumerate(val_dataloader):

            inputs = inputs.to(device)
            labels = labels.to(device)

            if device.type == "cuda":
                with autocast('cuda'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            y_pred = torch.sigmoid(outputs)
            val_y_pred_list.append(y_pred.cpu())
            val_y_true_list.append(labels.cpu())

            running_val_loss = running_val_loss + (loss.detach() * inputs.shape[0])
            total_samples += inputs.shape[0]

            pbar.update(1)

    pbar.close()

    all_y_true = torch.cat(val_y_true_list).numpy()
    all_y_pred = torch.cat(val_y_pred_list).numpy()

    avg_val_loss = (running_val_loss / total_samples).item()
    print(f"Epoch {epoch} Val Loss: ", avg_val_loss)
    val_losses.append({"epoch": epoch, "avg_val_loss": avg_val_loss})

    ranking_metrics = compute_ranking_metrics(all_y_true, all_y_pred)
    print(f"Epoch {epoch} VALIDATION RANKING:", ranking_metrics)
    val_ranking_stats.append({"epoch": epoch, **ranking_metrics})

    early_stopper(
        val_loss=avg_val_loss,
        epoch=epoch,
        model=model,
        opt=opt,
        extra_state={"val_losses": val_losses, "train_losses": train_losses}
    )

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "opt_state_dict": opt.state_dict(),
        "epoch": epoch+1,
        "lrs": lrs,
        "val_losses": val_losses,
        "train_losses": train_losses}
    torch.save(checkpoint, "checkpoint.pth")

    train_stats_df = pd.DataFrame(train_epoch_stats)
    train_stats_df.to_csv("train_epoch_stats.csv", index=False)
    train_losses_df = pd.DataFrame(train_losses)
    train_losses_df.to_csv("train_losses.csv", index=False)
    val_stats_df = pd.DataFrame(val_ranking_stats)
    val_stats_df.to_csv("val_ranking_stats.csv", index=False)
    val_losses_df = pd.DataFrame(val_losses)
    val_losses_df.to_csv("val_losses.csv", index=False)

    if early_stopper.early_stop:
        print("Early stopping triggered. Training halting.")
        break

For epoch 0, using learning rates {'backbone': 0.0, 'shared': 0.001, 'head': 0.001}


Epoch 0 - Training:  48%|████▊     | 179/376 [02:29<02:43,  1.21batch/s]

KeyboardInterrupt: 